# CoRe-TFM v7 — Standalone 160-Experiment Final Audit

**Input:** `CoReTFM_FULL160_Combined_Verified_Evidence.zip` (the verified combined evidence file). **No Google Drive, GPU, model checkpoint, or old Colab account needed.**

This notebook independently reconstructs fixed reconciliation, simple view-selection and validation-fitted calibration from the original saved `.npz` probabilities. It does **not** retrain either foundation model, does not silently trust earlier CSV summaries, does not unpickle the five processed-data files, and never claims to verify unknown checkpoint bytes.

**Paper-ready outputs:** validated 160-cache coverage, 10-dataset blocked inference and Holm correction, NLL/Brier/ECE/accuracy, direct hard/soft ablations, Diamonds class-pair mechanism analysis, consistency–accuracy figures, environment/provenance gaps, manuscript tables (`.tex`/`.csv`), evidence limitations, replication checklist, and a compact downloadable ZIP.

**Scientific protocol:** 10 datasets × 2 TFMs × 4 randomized split settings × 2 training caps. Primary comparison uses cap 256, 52 validation labels to choose policies, and 128 held-out test examples. The independent inferential unit is the **dataset**, not individual folds or prediction rows. No hypothesis is declared confirmatory from these exploratory analyses.

## 1. Configure input and local output

Upload the combined ZIP to Colab when prompted, or manually set `INPUT_ZIP`. Nothing is written to Google Drive. Output files are created under `/content/CoReTFM_v7_Final_Audit` on Colab. Keep the downloaded output ZIP as your backup.

In [ ]:
import os
from pathlib import Path

INPUT_ZIP = os.environ.get('COREFM_AUDIT_INPUT', '')  # Optional absolute ZIP path
STRICT_EXPECTED_160 = True
PRIMARY_TRAIN_CAP = 256
VALIDATION_SELECTION_BUDGET = 52
AUTO_UPLOAD_IN_COLAB = True
DOWNLOAD_AUDIT_ZIP = True
# Optional paths to current checkpoint files; hashes DO NOT prove original runs loaded them.
CHECKPOINT_FILES_TO_HASH = {}

IN_COLAB = 'google.colab' in __import__('sys').modules or Path('/content').is_dir()
BASE = Path('/content') if Path('/content').is_dir() else Path.cwd()
OUTPUT_ROOT = BASE / 'CoReTFM_v7_Final_Audit'
REPORT_DIR = OUTPUT_ROOT / 'reports'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print('Colab-like runtime:', IN_COLAB)
print('Output folder:', REPORT_DIR)


## 2. CPU requirements

This is an analysis notebook and deliberately does not install `torch`, `tabicl` or `tabpfn`. Those packages are unnecessary for verifying saved probabilities.

In [ ]:
import importlib.util, subprocess, sys
needed = {'numpy':'numpy','pandas':'pandas','scipy':'scipy','matplotlib':'matplotlib'}
missing = [pkg for name,pkg in needed.items() if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable,'-m','pip','install','-q',*missing],check=True)
print('CPU dependencies ready.')


## 3. Audit engine — archive and frozen-cache validation

The entire engine is embedded in this notebook and does not depend on another `.py` file. It checks the combined manifest, cache sidecars, original split hashes, class domains, per-view probability normalization, and split/test alignment.

In [ ]:
"""CoRe-TFM v7: standalone offline audit of saved TFM prediction views.
No downloaded packages, model weights, checkpoint claims, or Google Drive required.
The original evidence ZIP is not modified. Provenance status remains source-bounded.
"""
from __future__ import annotations
import os, re, io, sys, json, math, hashlib, zipfile, pathlib, warnings, platform
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon, spearmanr
import matplotlib.pyplot as plt

DATASETS = ('anneal','car','credit','customer','diamonds','marketing','mic','nursery','phishing','wine')
MODELS = ('tabiclv2','tabpfn3')
SPLITS = ((23,0),(23,1),(42,0),(71,0))
CAPS = (128,256)
EXPECTED_CACHE_COUNT = len(DATASETS)*len(MODELS)*len(SPLITS)*len(CAPS)
TEMPS = (0.65,0.8,1.0,1.25,1.5,2.0,3.0)
METHODS = ('arithmetic','geometric','raw_BA','raw_AB','hard','soft_25','soft_50','soft_75')
PROTOCOL = 'coretfm-v2-independent-fixed-test-2026-10-19'
SHA_RE = re.compile(r'^[0-9a-f]{64}$')


def sha(data): return hashlib.sha256(data).hexdigest()
def sha_obj(value): return sha(json.dumps(value, sort_keys=True, default=str).encode())
def write_json(path, value):
    path=pathlib.Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(json.dumps(value,sort_keys=True,indent=2,default=str),encoding='utf-8')


def audit_archive(zip_path, reportdir):
    """Byte-level, path-safe archive verification; does not extract or unpickle."""
    reportdir=pathlib.Path(reportdir);reportdir.mkdir(parents=True,exist_ok=True)
    exceptions=[]
    with zipfile.ZipFile(zip_path,'r') as z:
        infos=z.infolist(); names=[i.filename for i in infos]
        if len(names)!=len(set(names)):raise ValueError('Duplicated ZIP paths - ambiguous archive')
        for info in infos:
            path=pathlib.PurePosixPath(info.filename)
            if path.is_absolute() or '..' in path.parts or '\\' in info.filename:
                raise ValueError('Unsafe archive member '+info.filename)
            if info.file_size>500_000_000:
                raise ValueError('Oversized archive member '+info.filename)
        if z.testzip() is not None:raise ValueError('CRC corruption detected')
        manifest_name='COMBINED_SHA256_MANIFEST.json'
        if manifest_name not in names:raise ValueError('Missing combined SHA-256 manifest')
        manifest=json.loads(z.read(manifest_name))
        if not isinstance(manifest,list):raise ValueError('Bad manifest format')
        listed=set()
        for r in manifest:
            item=r['path']
            if item in listed:raise ValueError('Duplicate manifest path '+item)
            listed.add(item)
            if item not in names:raise ValueError('Manifest entry missing: '+item)
            b=z.read(item)
            if sha(b)!=r['sha256'] or len(b)!=r['bytes']:
                raise ValueError('Manifest hash/length mismatch: '+item)
        extra=set(names)-listed-{manifest_name}
        if extra:exceptions.append({'type':'unmanifested_members','count':len(extra),'examples':sorted(extra)[:15]})
        n_npz=sum(1 for n in names if n.startswith('real_caches/') and n.endswith('.npz'))
        raw_files=[n for n in names if n.startswith('datasets/') and n.endswith('.pkl')]
        processed_hashes={pathlib.PurePosixPath(n).name:sha(z.read(n)) for n in raw_files}
        status={'zip_name':pathlib.Path(zip_path).name,'zip_sha256':sha(pathlib.Path(zip_path).read_bytes()),
                'manifest_entries_verified':len(manifest),'archive_entries':len(names),
                'npz_count':n_npz,'processed_dataset_byte_hashes_NO_UNPICKLING':processed_hashes,
                'raw_openml_versions_verified':False,'zip_integrity':'PASS', 'exceptions':exceptions}
    write_json(reportdir/'01_ARCHIVE_INTEGRITY.json',status)
    return status


def cache_checks(zip_path, reportdir):
    """Validate saved model outputs, split containment, repeated tests and provenance IDs."""
    rows=[];issues=[];identities=set(); split_maps={}; ids_maps={}; id_models=defaultdict(dict)
    with zipfile.ZipFile(zip_path) as z:
        names=set(z.namelist())
        for path in sorted(n for n in names if n.startswith('splits/') and n.endswith('.json')):
            try:
                s=json.loads(z.read(path))
                key=(s['dataset'],int(s['seed']),int(s['fold']))
                if key in split_maps:raise ValueError('duplicate split identity')
                for col in ['train_pool_ids','val_ids','test_ids']:
                    ids=s[col]
                    if len(ids)!=len(set(ids)):raise ValueError(col+' duplicated')
                a,b,c=[set(s[k]) for k in ['train_pool_ids','val_ids','test_ids']]
                if a&b or a&c or b&c:raise ValueError('split overlap')
                if sha_obj({k:v for k,v in s.items() if k!='split_sha256'})!=s['split_sha256']:
                    raise ValueError('split hash invalid')
                if s['protocol']!=PROTOCOL:raise ValueError('protocol mismatch')
                split_maps[key]=s
            except Exception as e:issues.append({'file':path,'error':str(e)})
        for npz_name in sorted(n for n in names if n.startswith('real_caches/') and n.endswith('.npz')):
            try:
                side=npz_name[:-4]+'.json'
                if side not in names:raise ValueError('missing sidecar')
                m=json.loads(z.read(side));b=z.read(npz_name)
                if sha(b)!=m.get('npz_sha256'):raise ValueError('NPZ checksum mismatch')
                if m['protocol']!=PROTOCOL or m['artifact_kind']!='real_tfm':raise ValueError('wrong protocol/evidence kind')
                key=(m['dataset'],m['model'],int(m['seed']),int(m['fold']),int(m['train_cap']))
                if key in identities:raise ValueError('duplicated model configuration')
                identities.add(key)
                s=split_maps[(key[0],key[2],key[3])]
                if m['split_sha256']!=s['split_sha256']:raise ValueError('split reference mismatch')
                with np.load(io.BytesIO(b),allow_pickle=False) as d:
                    expected=('a_val','b_val','a_given_b_val','b_given_a_val','a_test','b_test','a_given_b_test','b_given_a_test','ya_val','yb_val','ya_test','yb_test','train_ids','val_ids','test_ids')
                    if set(expected)-set(d.files):raise ValueError('missing arrays')
                    nva= len(d['ya_val']);nte=len(d['ya_test']);ka=len(m['classes_a']);kb=len(m['classes_b'])
                    shapes={
                        'a_val':(nva,ka),'a_test':(nte,ka),'b_val':(nva,kb),'b_test':(nte,kb),
                        'a_given_b_val':(nva,kb,ka),'a_given_b_test':(nte,kb,ka),
                        'b_given_a_val':(nva,ka,kb),'b_given_a_test':(nte,ka,kb)}
                    for field,shape in shapes.items():
                        p=d[field]
                        if p.shape!=shape or not np.isfinite(p).all() or (p < -1e-8).any() or not np.allclose(p.sum(axis=-1),1,atol=1e-4):
                            raise ValueError('invalid probability shape, range or normalization: '+field)
                    for suffix,n in [('val',nva),('test',nte)]:
                        for f,k in [('ya_'+suffix,ka),('yb_'+suffix,kb)]:
                            v=d[f]
                            if len(v)!=n or (v<0).any() or (v>=k).any():raise ValueError('invalid label '+f)
                    tr=d['train_ids'].tolist();val=d['val_ids'].tolist();tst=d['test_ids'].tolist()
                    if tr!=s['train_pool_ids'][:key[4]]:raise ValueError('train IDs are not split prefix')
                    if val!=s['val_ids'] or tst!=s['test_ids']:raise ValueError('validation/test IDs differ from frozen split')
                    if any(sha_obj(ids)!=m[k] for ids,k in [(tr,'train_ids_sha'),(val,'val_ids_sha'),(tst,'test_ids_sha')]):
                        raise ValueError('ID checksum mismatch')
                    if nva!=int(m['n_val']) or nte!=int(m['n_test']) or len(tr)!=int(m['n_train']):raise ValueError('row-count mismatch')
                    ids_maps[key]=(tr,val,tst)
                    id_models[(key[0],key[2],key[3],key[4])][key[1]]=(tr,val,tst)
                    rows.append({'dataset':key[0],'model':key[1],'seed':key[2],'fold':key[3], 'train_cap':key[4],
                                 'n_train':len(tr),'n_val':nva,'n_test':nte,'ka':ka,'kb':kb,
                                 'npz_sha256':m['npz_sha256'],'split_sha256':m['split_sha256'],
                                 'checkpoint_identifier':m.get('checkpoint_identifier',''),
                                 'checkpoint_sha256':m.get('checkpoint_sha256',''),
                                 'checkpoint_revision_confirmed':m.get('checkpoint_revision_confirmed',False),
                                 'package_version':m.get('package_version',''),
                                 'gpu':m.get('device',{}).get('gpu'),
                                 'cuda':m.get('device',{}).get('cuda'),
                                 'torch':m.get('device',{}).get('torch'),
                                 'elapsed_s':m.get('elapsed_s'),
                                 'openml_source':json.dumps(m.get('source',{}),sort_keys=True),
                                 'cache_path':npz_name})
            except Exception as e:issues.append({'file':npz_name,'error':repr(e)})
    # fixed model view comparison on exactly matching splits, with nested training contexts
    for k,v in id_models.items():
        if len(v)==2:
            if v['tabiclv2']!=v['tabpfn3']:issues.append({'file':str(k),'error':'cross-model split IDs differ'})
    for ds in DATASETS:
        for sd,fl in SPLITS:
            for mod in MODELS:
                k1=(ds,mod,sd,fl,128);k2=(ds,mod,sd,fl,256)
                if k1 in ids_maps and k2 in ids_maps:
                    a,b=ids_maps[k1],ids_maps[k2]
                    if a[0]!=b[0][:128] or a[1]!=b[1] or a[2]!=b[2]:
                        issues.append({'file':str(k1),'error':'training sizes not nested or held-out IDs drifted'})
    out=pd.DataFrame(rows)
    out.to_csv(pathlib.Path(reportdir)/'02_CACHE_PROVENANCE.csv',index=False)
    counts={'status':'PASS' if not issues and len(rows)==EXPECTED_CACHE_COUNT else 'INCOMPLETE_OR_INVALID',
            'valid_caches':len(rows),'expected_caches':EXPECTED_CACHE_COUNT,
            'datasets':sorted(out.dataset.unique().tolist()) if len(out) else [],
            'models':sorted(out.model.unique().tolist()) if len(out) else [],
            'split_specs':len(split_maps),'expected_split_specs':len(DATASETS)*len(SPLITS),
            'issues':issues,'checkpoint_weights_verified':False,
            'note':'Cache hashes verify archived predictions, not checkpoint weights or original raw OpenML files.'}
    write_json(pathlib.Path(reportdir)/'02_CACHE_AUDIT_STATUS.json',counts)
    return counts,out




## 4. Locate or upload the combined evidence ZIP

In [ ]:
import zipfile
if INPUT_ZIP:
    archive_path = Path(INPUT_ZIP).expanduser()
else:
    lookup = [BASE/'CoReTFM_FULL160_Combined_Verified_Evidence.zip',
              Path.cwd()/'CoReTFM_FULL160_Combined_Verified_Evidence.zip',
              Path('/mnt/data/CoReTFM_FULL160_Combined_Verified_Evidence.zip')]
    archive_path = next((p for p in lookup if p.is_file()), None)
    if archive_path is None and AUTO_UPLOAD_IN_COLAB:
        try:
            from google.colab import files
            uploaded = files.upload()
            if not uploaded:
                raise FileNotFoundError('Nothing uploaded. Upload the COMBINED 160-cache ZIP.')
            possible = [name for name in uploaded if name.lower().endswith('.zip')]
            if len(possible)!=1:
                raise ValueError('Upload exactly one combined ZIP, not multiple snapshots.')
            archive_path = Path(possible[0])
        except ImportError:
            pass
if archive_path is None or not archive_path.is_file():
    raise FileNotFoundError('Combined evidence ZIP not found. Upload CoReTFM_FULL160_Combined_Verified_Evidence.zip or set INPUT_ZIP.')
if not zipfile.is_zipfile(archive_path):
    raise ValueError(f'Not a valid ZIP file: {archive_path}')
print('Using evidence file:', archive_path, '|', f'{archive_path.stat().st_size/1048576:.2f} MiB')


## 5. Validate the archive-wide manifest and dataset snapshots

The stored `.pkl` files are only **hashed as bytes**. They are never unpickled. Their hashes cannot substitute for original OpenML source-file revisions and checksums.

In [ ]:
archive_status = audit_archive(archive_path, REPORT_DIR)
print('Archive SHA-256:', archive_status['zip_sha256'])
print('Manifest entries verified:', archive_status['manifest_entries_verified'])
print('Prediction cache ZIP entries:', archive_status['npz_count'])
print('Processed dataset snapshots:', len(archive_status['processed_dataset_byte_hashes_NO_UNPICKLING']))
assert archive_status['zip_integrity']=='PASS'


## 6. Validate all 160 real-model configurations and paired frozen splits

In [ ]:
cache_status, cache_df = cache_checks(archive_path, REPORT_DIR)
print('Cache audit:',cache_status['status'])
print('Verified caches:',cache_status['valid_caches'],'/',cache_status['expected_caches'])
print('Verified split specifications:',cache_status['split_specs'])
print('Errors:',len(cache_status['issues']))
if STRICT_EXPECTED_160 and cache_status['status']!='PASS':
    raise RuntimeError('Cache integrity/completeness failed. See 02_CACHE_AUDIT_STATUS.json. Do not report this as a complete 160-config benchmark.')
from IPython.display import display
display(cache_df.groupby(['model','dataset','train_cap']).size().rename('valid_caches').reset_index().head(20))


## 7. Reconstruct fixed interventions and validation-only selectors

Fixed interventions: arithmetic/geometric pooling, both original factorization orders, hard marginal projection, and 25/50/75% soft projections. Validation-only methods: raw-or-pool, full family, temperature-scaled pool and a conservative repair gate. The method family is exploratory, and the penalty is **not optimized using test labels**.

In [ ]:
def normalize(x):
    x=np.maximum(np.asarray(x,dtype=np.float64),1e-12)
    return x/x.sum(axis=(1,2),keepdims=True)
def implied(d,s):
    a=d['a_'+s];b=d['b_'+s]
    ba=normalize(np.transpose(d['a_given_b_'+s],(0,2,1))*b[:,None,:])
    ab=normalize(a[:,:,None]*d['b_given_a_'+s])
    return ba,ab

def hard_projection(j,pa,pb,max_iter=150,tol=1e-9):
    p=normalize(j);pa=np.maximum(np.array(pa,copy=True),1e-12); pb=np.maximum(np.array(pb,copy=True),1e-12)
    pa=pa/pa.sum(axis=1,keepdims=True);pb=pb/pb.sum(axis=1,keepdims=True)
    for i in range(max_iter):
        p=p*(pa[:,:,None]/np.maximum(p.sum(axis=2,keepdims=True),1e-15))
        p=p*(pb[:,None,:]/np.maximum(p.sum(axis=1,keepdims=True),1e-15))
        if max(np.max(np.abs(p.sum(axis=2)-pa)),np.max(np.abs(p.sum(axis=1)-pb)))<tol:break
    return normalize(p)

def views(d,s):
    ba,ab=implied(d,s)
    pool=normalize(.5*ba+.5*ab)
    hp=hard_projection(pool,d['a_'+s],d['b_'+s])
    return {'arithmetic':pool,'geometric':normalize(np.sqrt(np.maximum(ba,1e-12)*np.maximum(ab,1e-12))),
            'raw_BA':ba,'raw_AB':ab,'hard':hp,
            'soft_25':normalize(.75*pool+.25*hp),'soft_50':normalize(.5*pool+.5*hp),
            'soft_75':normalize(.25*pool+.75*hp)}

def temp_scale(p,t):
    l=np.log(np.maximum(p,1e-12))/float(t)
    l-=l.max(axis=(1,2),keepdims=True)
    return normalize(np.exp(l))

def scores(p,ya,yb):
    n=len(ya);kb=p.shape[2];flat=p.reshape(n,-1)
    truth=np.asarray(ya,dtype=np.int64)*kb+np.asarray(yb,dtype=np.int64)
    phat=flat[np.arange(n),truth]
    nll=-np.log(np.maximum(phat,1e-12))
    brier=(flat**2).sum(axis=1)+1-2*phat
    pred=flat.argmax(axis=1)
    conf=flat.max(axis=1)
    bucket=np.minimum((conf*10).astype(int),9)
    ece=sum((bucket==i).mean()*abs((pred[bucket==i]==truth[bucket==i]).mean()-conf[bucket==i].mean())
            for i in range(10) if np.any(bucket==i))
    return {'nll':float(nll.mean()),'brier':float(brier.mean()),'accuracy':float((pred==truth).mean()),'ece_top1':float(ece)},nll

def marg_tv(p,pa,pb):
    va=.5*np.abs(p.sum(axis=2)-pa).sum(axis=1)
    vb=.5*np.abs(p.sum(axis=1)-pb).sum(axis=1)
    return float(np.mean((va+vb)/2))


def analyze_prediction_caches(zip_path, reportdir, primary_cap=256, val_budget=52):
    """Reconstruct fixed interventions and fit only predeclared selectors on validation labels."""
    rows=[];diamonds=[];errors=[];rdir=pathlib.Path(reportdir)
    for name in ['03_CONFIGURATION_METHODS.csv','03_DIAMONDS_CLASS_DIAGNOSTICS.csv']:
        p=rdir/name
        if p.exists():p.unlink()
    with zipfile.ZipFile(zip_path,'r') as z:
        items=sorted(n for n in z.namelist() if n.startswith('real_caches/') and n.endswith('.npz'))
        for name in items:
            try:
                m=json.loads(z.read(name[:-4]+'.json'))
                with np.load(io.BytesIO(z.read(name)),allow_pickle=False) as f:
                    d={k:f[k] for k in f.files}
                V=views(d,'val');T=views(d,'test')
                ya_v=d['ya_val'][:val_budget];yb_v=d['yb_val'][:val_budget]
                ya_t=d['ya_test'];yb_t=d['yb_test']
                val_loss={k:scores(p[:val_budget],ya_v,yb_v)[0]['nll'] for k,p in V.items()}
                raw_subset=('arithmetic','geometric','raw_BA','raw_AB')
                selected_raw=min(raw_subset,key=lambda s:val_loss[s])
                selected_full=min(METHODS,key=lambda s:val_loss[s])
                selT=min(TEMPS,key=lambda t:scores(temp_scale(V['arithmetic'][:val_budget],t),ya_v,yb_v)[0]['nll'])
                base_cal=temp_scale(T['arithmetic'],selT)
                # Selectively replace calibrated pooling only when held-out validation evidence clears a predeclared penalty.
                # This is a post-hoc exploratory policy; not a registered confirmatory method.
                repair_keys=('soft_25','soft_50','soft_75','hard')
                best_repair=min(repair_keys,key=lambda k:val_loss[k])
                cal_val_loss=scores(temp_scale(V['arithmetic'][:val_budget],selT),ya_v,yb_v)[0]['nll']
                threshold=.15*math.sqrt(math.log(len(repair_keys)+1)/val_budget)
                gated=(best_repair if cal_val_loss-val_loss[best_repair]>threshold else 'CALIBRATED_FALLBACK')
                valTV=.5*np.abs(V['raw_BA']-V['raw_AB']).sum(axis=(1,2))
                testTV=.5*np.abs(T['raw_BA']-T['raw_AB']).sum(axis=(1,2))
                baseline_marg=marg_tv(T['arithmetic'],d['a_test'],d['b_test'])
                hard_marg=marg_tv(T['hard'],d['a_test'],d['b_test'])
                # Gradients/fit are NOT performed using test labels; only scores are computed on test.
                methods={k:p for k,p in T.items()}
                methods.update({'raw_or_pool':T[selected_raw], 'full_family':T[selected_full],
                                'temperature_pool':base_cal,
                                'validation_guarded_repair':base_cal if gated=='CALIBRATED_FALLBACK' else T[gated]})
                for method,pred in methods.items():
                    met,_=scores(pred,ya_t,yb_t)
                    rows.append({'dataset':m['dataset'],'model':m['model'],'seed':int(m['seed']),
                                 'fold':int(m['fold']),'train_cap':int(m['train_cap']),
                                 'validation_budget':val_budget,'n_test':len(ya_t),'method':method,
                                 **met,'marginal_tv':marg_tv(pred,d['a_test'],d['b_test']),
                                 'factorization_tv':float(testTV.mean()),
                                 'pool_to_hard_marginal_tv_reduction':baseline_marg-hard_marg,
                                 'selector_raw':selected_raw,'selector_full':selected_full,
                                 'temperature_selected':selT,'guard_action':gated,
                                 'guard_penalty':threshold, 'cache_file':name})
                if m['dataset']=='diamonds' and m['train_cap']==primary_cap:
                    for class_a in np.unique(ya_t):
                        for class_b in np.unique(yb_t):
                            idx=(ya_t==class_a)&(yb_t==class_b)
                            if not idx.any():continue
                            loss_pool=-np.log(np.maximum(T['arithmetic'][idx,class_a,class_b],1e-12))
                            loss_hard=-np.log(np.maximum(T['hard'][idx,class_a,class_b],1e-12))
                            diff=loss_hard-loss_pool
                            diamonds.append({'dataset':'diamonds','model':m['model'],'seed':m['seed'],'fold':m['fold'],
                                             'class_a':m['classes_a'][int(class_a)],'class_b':m['classes_b'][int(class_b)],
                                             'n_test_class_pair':int(idx.sum()),'delta_nll_hard_vs_pool':float(diff.mean()),
                                             'fraction_test_examples_harmed':float((diff>0).mean()),
                                             'mean_factor_disagreement':float(testTV[idx].mean())})
            except Exception as e:errors.append({'file':name,'error':repr(e)})
    df=pd.DataFrame(rows)
    if len(df):df.to_csv(rdir/'03_CONFIGURATION_METHODS.csv',index=False)
    pd.DataFrame(diamonds).to_csv(rdir/'03_DIAMONDS_CLASS_DIAGNOSTICS.csv',index=False)
    status={'status':'PASS' if not errors and len(df)>0 else 'FAILED_OR_INCOMPLETE',
            'cache_configs':int(df[['dataset','model','seed','fold','train_cap']].drop_duplicates().shape[0]) if len(df) else 0,
            'policy_rows':len(df),'methods':sorted(df.method.unique().tolist()) if len(df) else [],
            'errors':errors,
            'selection_rule':'validation-only, first 52 labels; no test labels used to fit selector or temperature',
            'guard_policy':'exploratory prespecified in notebook; not a confirmatory pretrained repair policy'}
    write_json(rdir/'03_ANALYSIS_STATUS.json',status)
    return df,status




In [ ]:
method_df, method_status = analyze_prediction_caches(
    archive_path, REPORT_DIR, primary_cap=PRIMARY_TRAIN_CAP,
    val_budget=VALIDATION_SELECTION_BUDGET)
print('Recomputed configurations:',method_status['cache_configs'])
print('Method-evaluation rows:',method_status['policy_rows'])
print('Methods:',', '.join(method_status['methods']))
if STRICT_EXPECTED_160 and (method_status['status']!='PASS' or method_status['cache_configs']!=160):
    raise RuntimeError('Failed to reconstruct every cache. Inspect 03_ANALYSIS_STATUS.json.')


## 8. Ten-dataset inference and multiplicity correction

**Caution:** Wilcoxon comparisons use 10 paired dataset means, with repeated split settings collapsed inside each dataset. These are exploratory tests; p-values are given both unadjusted and Holm-adjusted across a declared family of seven comparisons within each model. Confidence intervals are dataset-bootstrap summaries, not guaranteed coverage under arbitrary dataset shift.

In [ ]:
def summarize_by_dataset(df, reportdir, primary_cap=256):
    """Aggregate within independent dataset blocks; splits are repeated observations."""
    rdir=pathlib.Path(reportdir)
    d=df.loc[df.train_cap==primary_cap].copy()
    key=['dataset','model','seed','fold','train_cap']
    # Exact pairing: all methods have the same configuration IDs.
    if d.duplicated(key+['method']).any():raise ValueError('Duplicate method per configuration')
    n_methods=d.method.nunique()
    if (d.groupby(key).method.nunique()!=n_methods).any():raise ValueError('Unmatched method coverage')
    base=d[d.method=='arithmetic'][key+['nll','brier','accuracy','ece_top1','marginal_tv']].rename(
        columns={s:'baseline_'+s for s in ['nll','brier','accuracy','ece_top1','marginal_tv']})
    paired=d.merge(base,on=key,validate='many_to_one')
    for metric in ['nll','brier','accuracy','ece_top1','marginal_tv']:
        paired['delta_'+metric]=paired[metric]-paired['baseline_'+metric]
    paired.to_csv(rdir/'04_PAIRED_CONFIGURATION_EFFECTS.csv',index=False)
    aggregations={k:(k,'mean') for k in ['nll','brier','accuracy','ece_top1','marginal_tv','delta_nll','delta_brier','delta_accuracy','delta_ece_top1','delta_marginal_tv']}
    block=paired.groupby(['model','dataset','method'],as_index=False).agg(**aggregations,split_count=('seed','size'))
    block.to_csv(rdir/'04_DATASET_BLOCKS.csv',index=False)
    contrasts=['hard','soft_50','raw_or_pool','full_family','temperature_pool','validation_guarded_repair']
    # Comparisons against pre-specified comparator; guard is compared with calibrated pool too.
    temp_ref=block[block.method=='temperature_pool'][['model','dataset','nll']].rename(columns={'nll':'calibrated_pool_nll'})
    block=block.merge(temp_ref,on=['model','dataset'],validate='many_to_one')
    block['delta_nll_vs_calibrated_pool']=block.nll-block.calibrated_pool_nll
    rng=np.random.default_rng(20261020)
    tests=[]
    for model in MODELS:
        g=block[block.model==model]
        for method in contrasts:
            sub=g[g.method==method].sort_values('dataset')
            arr=sub.delta_nll.to_numpy()
            if len(arr)!=len(DATASETS):
                raise ValueError(f'Only {len(arr)} independent datasets for {model}/{method}')
            boot=np.mean(rng.choice(arr,(10000,len(arr)),replace=True),axis=1)
            ci=np.quantile(boot,[.025,.975])
            p=1.0 if np.allclose(arr,0) else float(wilcoxon(arr,alternative='two-sided',zero_method='wilcox').pvalue)
            tests.append({'model':model,'method':method,'reference':'arithmetic','n_dataset_blocks':len(arr),
                          'mean_delta_nll':float(arr.mean()),'mean_delta_brier':float(sub.delta_brier.mean()),
                          'bootstrap_95_low':float(ci[0]),'bootstrap_95_high':float(ci[1]),
                          'dataset_wins_nll':int((arr<0).sum()),'dataset_harms_nll':int((arr>0).sum()),
                          'wilcoxon_two_sided_p_exploratory':p})
        # Additional comparison of gated repair vs calibrated pooling
        sub=g[g.method=='validation_guarded_repair'].sort_values('dataset')
        arr=sub.delta_nll_vs_calibrated_pool.to_numpy()
        boot=np.mean(rng.choice(arr,(10000,len(arr)),replace=True),axis=1)
        ci=np.quantile(boot,[.025,.975])
        p=1.0 if np.allclose(arr,0) else float(wilcoxon(arr,alternative='two-sided',zero_method='wilcox').pvalue)
        tests.append({'model':model,'method':'validation_guarded_repair','reference':'temperature_pool','n_dataset_blocks':len(arr),
                      'mean_delta_nll':float(arr.mean()),'mean_delta_brier':np.nan,
                      'bootstrap_95_low':float(ci[0]),'bootstrap_95_high':float(ci[1]),
                      'dataset_wins_nll':int((arr<0).sum()),'dataset_harms_nll':int((arr>0).sum()),
                      'wilcoxon_two_sided_p_exploratory':p})
    res=pd.DataFrame(tests)
    # Adjust within each model over all exploratory tests, including the gated comparison.
    res['holm_p_exploratory']=np.nan
    for mod in MODELS:
        ix=res.index[res.model==mod].tolist()
        ordered=sorted(ix,key=lambda i:res.loc[i,'wilcoxon_two_sided_p_exploratory'])
        running=0
        for j,i in enumerate(ordered):
            running=max(running,min(1,(len(ordered)-j)*float(res.loc[i,'wilcoxon_two_sided_p_exploratory'])))
            res.loc[i,'holm_p_exploratory']=running
    res.to_csv(rdir/'04_DATASET_BLOCKED_UNCERTAINTY.csv',index=False)
    return paired,block,res




In [ ]:
paired, dataset_blocks, tests = summarize_by_dataset(method_df, REPORT_DIR, primary_cap=PRIMARY_TRAIN_CAP)
columns=['model','method','reference','mean_delta_nll','bootstrap_95_low','bootstrap_95_high','dataset_harms_nll','wilcoxon_two_sided_p_exploratory','holm_p_exploratory']
display(tests[columns].round(6))
print('Independent data-set blocks:',dataset_blocks.dataset.nunique(),'; matched primary method rows:',len(paired))


## 9. Mechanism figures, Diamonds exception and recorded provenance

The Diamonds exception is studied at split and class-pair levels. Sparse class pairs are reported with sample counts. We do not infer a causal mechanism from correlation between marginal-TV reduction and NLL change.

In [ ]:
def figures_and_diagnostics(df,paired,blocks,reportdir,primary_cap=256):
    rdir=pathlib.Path(reportdir)
    focus=blocks[(blocks.method.isin(['hard','raw_or_pool','full_family','temperature_pool','arithmetic']))]
    for model in MODELS:
        g=focus[focus.model==model]
        pivot=g.pivot(index='dataset',columns='method',values='nll').reindex(DATASETS)
        ax=pivot.plot(kind='bar',figsize=(13,5),width=.8)
        ax.set_title(f'{model}: joint test NLL by dataset, 256 training examples')
        ax.set_ylabel('NLL (lower is better)');ax.set_xlabel('Dataset');ax.legend(ncol=3,fontsize=8)
        plt.tight_layout();plt.savefig(rdir/f'05_NLL_DATASETS_{model}.png',dpi=170);plt.close()
    hard=blocks[blocks.method=='hard'].pivot(index='dataset',columns='model',values='delta_nll').reindex(DATASETS)
    ax=hard.plot(kind='bar',figsize=(12,5));ax.axhline(0,color='black',linewidth=.7)
    ax.set_title('Direct hard reconciliation versus arithmetic pooling')
    ax.set_ylabel('Difference in test NLL (positive = harm)');ax.set_xlabel('Dataset')
    plt.tight_layout();plt.savefig(rdir/'05_HARD_REPAIR_NLL_DELTA.png',dpi=180);plt.close()
    for model in MODELS:
        sub=paired[(paired.model==model)&(paired.method=='hard')]
        g=sub.groupby('dataset',as_index=False).agg(marginal_tv_reduction=('delta_marginal_tv',lambda v:float(-v.mean())),
                                                     nll_change=('delta_nll','mean'))
        fig,ax=plt.subplots(figsize=(8,5))
        ax.scatter(g.marginal_tv_reduction,g.nll_change)
        for _,r in g.iterrows():ax.annotate(r.dataset,(r.marginal_tv_reduction,r.nll_change),xytext=(4,4),textcoords='offset points',fontsize=8)
        ax.axhline(0,color='black',linewidth=.7)
        ax.set_title(f'{model}: coherence improvement is not guaranteed accuracy improvement')
        ax.set_xlabel('Marginal TV reduction from hard projection');ax.set_ylabel('Hard minus pool test NLL')
        plt.tight_layout();plt.savefig(rdir/f'05_CONSISTENCY_ACCURACY_{model}.png',dpi=170);plt.close()
    # Quantify Diamonds behavior at split vs dataset level, avoid ecological inference.
    focus_hard=paired[paired.method=='hard']
    diam=focus_hard[focus_hard.dataset=='diamonds'].copy()
    diamonds_status=diam[['model','seed','fold','delta_nll','delta_brier','delta_marginal_tv']].sort_values(['model','seed','fold'])
    diamonds_status.to_csv(rdir/'05_DIAMONDS_SPLIT_DETAILS.csv',index=False)
    flag={
        'n_datasets':len(DATASETS),'hard_negative_transfer_count':int((focus_hard.delta_nll>0).sum()),
        'hard_benefit_count':int((focus_hard.delta_nll<0).sum()),
        'hard_consistency_improvement_count':int((focus_hard.delta_marginal_tv<0).sum()),
        'comparison_unit':'dataset (four repeated split settings averaged before inferential comparisons)',
        'diamonds_note':'Observed dataset exception is descriptive; class-pair diagnostic sample counts can be sparse.'}
    write_json(rdir/'05_MECHANISM_SUMMARY.json',flag)
    return flag


def provenance_report(audit_cache, cache_df, reportdir, additional_checkpoint_paths=None):
    rdir=pathlib.Path(reportdir)
    pkg=(cache_df.groupby('model').agg(package_versions=('package_version',lambda s:sorted(set(s))),
                                          checkpoint_identifiers=('checkpoint_identifier',lambda s:sorted(set(s))),
                                          recorded_weight_hashes=('checkpoint_sha256',lambda s:sorted(set(s))),
                                          gpu_names=('gpu',lambda s:sorted(set(map(str,s)))),
                                          torch_versions=('torch',lambda s:sorted(set(map(str,s))))).reset_index())
    pkg.to_csv(rdir/'06_ENVIRONMENT_PROVENANCE.csv',index=False)
    user_paths=[]
    for desc,value in (additional_checkpoint_paths or {}).items():
        path=pathlib.Path(value)
        user_paths.append({'claimed_checkpoint':desc,'provided_path':str(path),'exists':path.is_file(),
                           'file_sha256_if_supplied':sha(path.read_bytes()) if path.is_file() else None,
                           'linked_to_loaded_model_during_original_run':False})
    caveats=[
      'Saved NPZ and JSON hashes verify archived predictions, not exact TFM weights loaded.',
      'The original model sessions recorded checkpoint identifiers but not authenticated checkpoint weight bytes.',
      'Five processed dataset pickle files are included, but they are NOT the original OpenML source bytes; never unpickle an untrusted archive.',
      'No raw OpenML revisions or dataset-download SHA-256 records are independently verified.',
      'Diamonds required a pairing/target correction between notebook generations; transformed feature schema and raw OpenML provenance require separate review.',
      'Four repeated randomized train/validation/test splits per dataset are correlated. Statistical unit is dataset.',
      'Prediction cache scores cannot prove original training code and package checkpoints are reproducible without a clean rerun.',
      'The original runs used Colab GPUs; recorded fit+predict timing is not a controlled steady-state latency benchmark.',
      'Method-family selection and calibration are trained on 52 validation labels; test is reserved for one-time evaluation.',
      'Fixed 205 validation and 128 test examples alongside 128/256 training examples are intentionally decoupled, not total-label budgets.'
    ]
    report={'audit_cache_complete':audit_cache['status']=='PASS',
            'n_caches':len(cache_df),'model_weight_sha256_verified':False,
            'original_raw_openml_byte_hashes_verified':False,
            'original_raw_openml_revision_provenance_verified':False,
            'candidate_weight_files':user_paths,
            'candidate_weight_warning':'A hash supplied now does not prove a historical run loaded those bytes.',
            'critical_limitations':caveats,
            'environment_at_analysis':{'python':sys.version.split()[0],'platform':platform.platform(),
                                        'numpy':np.__version__,'pandas':pd.__version__},
            'journal_ready_without_more_work':False}
    write_json(rdir/'06_PROVENANCE_GAPS.json',report)
    return report




In [ ]:
mechanism = figures_and_diagnostics(method_df, paired, dataset_blocks, REPORT_DIR, primary_cap=PRIMARY_TRAIN_CAP)
provenance = provenance_report(cache_status, cache_df, REPORT_DIR,
                               additional_checkpoint_paths=CHECKPOINT_FILES_TO_HASH)
print('Hard repair increased NLL in',mechanism['hard_negative_transfer_count'],'/',
      mechanism['hard_negative_transfer_count']+mechanism['hard_benefit_count'],'primary paired configurations')
print('Hard repair reduced marginal discrepancy in',mechanism['hard_consistency_improvement_count'],'configurations')
print('Exact historical checkpoint weights verified:',provenance['model_weight_sha256_verified'])
display(pd.read_csv(REPORT_DIR/'05_DIAMONDS_SPLIT_DETAILS.csv'))


## 10. Manuscript tables, explicit limitations, and replication checklist

The LaTeX table and results notes are **working manuscript materials**, not proof of journal readiness. The results text deliberately avoids claiming that the guarded repair method demonstrated useful activation.

In [ ]:
def export_manuscript_tables(blocks,stats,mechanism,reportdir):
    rdir=pathlib.Path(reportdir)
    order=['arithmetic','raw_or_pool','full_family','temperature_pool','hard','soft_50','validation_guarded_repair']
    vals=blocks.groupby(['model','method'],as_index=False).agg(mean_nll=('nll','mean'),mean_brier=('brier','mean'),
                                                            mean_acc=('accuracy','mean'),mean_ece=('ece_top1','mean'))
    vals=vals[vals.method.isin(order)].copy()
    vals['method']=pd.Categorical(vals.method,categories=order,ordered=True)
    vals=vals.sort_values(['method','model'])
    vals.to_csv(rdir/'07_MAIN_RESULTS_TABLE.csv',index=False)
    pivot=vals.pivot(index='method',columns='model',values='mean_nll').reindex(order)
    def esc(s):
        return str(s).replace('\\',r'\textbackslash{}').replace('_',r'\_').replace('%',r'\%').replace('&',r'\&')
    tex=[r'\begin{table}[t]',r'\centering',r'\caption{Ten-dataset mean test negative log-likelihood at 256 training examples and 52 validation labels. Each dataset has four paired randomized splits; lower is better.}',r'\label{tab:coretfm-primary}',r'\begin{tabular}{lrr}',r'\hline',r'Method & TabICLv2 & TabPFN-3 \\',r'\hline']
    for method,row in pivot.iterrows():
        tex.append(f'{esc(method)} & {row["tabiclv2"]:.5f} & {row["tabpfn3"]:.5f} '+r'\\')
    tex.extend([r'\hline',r'\end{tabular}',r'\end{table}'])
    (rdir/'07_MAIN_TABLE.tex').write_text('\n'.join(tex),encoding='utf-8')
    p=stats[(stats.method=='hard')&(stats.reference=='arithmetic')]
    para=['## CoRe-TFM 160-cache audited result notes', '',
          '**Scope:** 10 tabular datasets, TabICLv2 and TabPFN-3, 4 randomized held-out split settings per dataset, 128/256 training examples, 205 validation examples (52 used for primary selection), and 128 test examples.',
          '', '**Main result:** Hard projection reduces marginal inconsistency on the saved predictions, but on average worsens test NLL. This finding does not establish that all coherent postprocessing is harmful.',
          '', '**Primary fixed hard-projection contrasts:**']
    for _,r in p.iterrows():
        para.append(f'- {r.model}: mean dataset-blocked NLL difference {r.mean_delta_nll:+.5f}, bootstrap 95% [{r.bootstrap_95_low:+.5f}, {r.bootstrap_95_high:+.5f}], {r.dataset_harms_nll}/10 datasets harmed, exploratory two-sided Wilcoxon p={r.wilcoxon_two_sided_p_exploratory:.5f}, exploratory Holm p={r.holm_p_exploratory:.5f}.')
    para += ['', '**Interpretation cautions:** Do not infer significance from the 80 split settings. Calibration, candidate selection and gating must be treated as validation-selected methods rather than superior because of individual held-out outcomes.',
             '**Mechanism:** Compare marginal-TV reduction with the paired NLL and Brier difference. Diamonds is a heterogeneous exception; examine the provided split-level and class-pair diagnostics, including sparse cells.',
             '**Provenance:** Exact loaded checkpoint-weight SHA-256 and raw OpenML revision/bytes remain unverified. Processed pickle hashes are not substitutes for raw dataset hashes.',
             '**Submission gate:** Completed saved-prediction audit is not a clean independent rerun of the trained TFMs. An independent pinned-environment rerun and methods documentation are still needed.',
             '', 'All numerical entries are independently computed from raw saved probability arrays. Review every interpretation before manuscript insertion.']
    (rdir/'07_PAPER_RESULTS_AND_LIMITATIONS.md').write_text('\n'.join(para),encoding='utf-8')
    # Explicit reproducibility checklist for authors, not unverified claims
    checklist=['# Clean independent replication checklist',
      '- [ ] Obtain and retain original OpenML dataset IDs AND dataset revisions/download hashes, not merely processed pickle hashes.',
      '- [ ] Document every feature transformation, missing-value rule, label taxonomy, and the Diamonds cut/color pairing correction.',
      '- [ ] Pin Python, NumPy, pandas, scikit-learn, tabicl, tabpfn, PyTorch, CUDA, and checkpoint revisions.',
      '- [ ] Record SHA-256 for **exact bytes loaded by each original model**, including file paths or repository commits.',
      '- [ ] Independently rerun at least Car, Nursery, and Diamonds for both TFMs at the primary 256 context.',
      '- [ ] Retain and compare per-sample prediction arrays; declare numerical tolerance before comparison.',
      '- [ ] Keep splits and 52 validation labels fixed; do not tune reported methods on test labels.',
      '- [ ] Separate inferences supported by saved caches from those supported only after model retraining.',
      '- [ ] Disclose dataset-level exploratory multiplicity adjustment and 10-dataset inferential limitations.',
      '- [ ] Verify journal policy, licensing, model-weight sharing restrictions, and disclosure wording.'
      ]
    (rdir/'07_REPLICATION_CHECKLIST.md').write_text('\n'.join(checklist),encoding='utf-8')
    return vals


def make_archive(reportdir):
    rdir=pathlib.Path(reportdir)
    dest=rdir.parent/'CoReTFM_v7_FINAL_PAPER_AUDIT.zip'
    items=sorted(f for f in rdir.rglob('*') if f.is_file() and f.suffix.lower() not in ('.zip','.pkl','.npz'))
    manifest=[]
    with zipfile.ZipFile(dest,'w',zipfile.ZIP_DEFLATED,compresslevel=6) as z:
        for f in items:
            rel=f.relative_to(rdir).as_posix();b=f.read_bytes();z.writestr(rel,b)
            manifest.append({'path':rel,'sha256':sha(b),'bytes':len(b)})
        z.writestr('FINAL_AUDIT_SHA256_MANIFEST.json',json.dumps(manifest,indent=2))
    with zipfile.ZipFile(dest) as z:
        if z.testzip():raise ValueError('Export ZIP corrupted')
        for r in manifest:
            b=z.read(r['path']);assert sha(b)==r['sha256'] and len(b)==r['bytes']
    return dest,len(items)


In [ ]:
main_table = export_manuscript_tables(dataset_blocks,tests,mechanism,REPORT_DIR)
display(main_table.round(5))
print('LaTeX table:',REPORT_DIR/'07_MAIN_TABLE.tex')
print('Paper notes:',REPORT_DIR/'07_PAPER_RESULTS_AND_LIMITATIONS.md')
print('Replication checklist:',REPORT_DIR/'07_REPLICATION_CHECKLIST.md')


## 11. Cross-check independently recomputed scores with the prior archive summary

This is a *secondary* check only. The notebook first reconstructs its own scores from the 160 raw prediction arrays; it does not derive results by copying the previous CSV.

In [ ]:
with zipfile.ZipFile(archive_path) as z:
    previous = pd.read_csv(z.open('independent_audit/CoReTFM_160_Independent_Configuration_Results.csv'))
recomputed = (method_df[method_df.method.isin(['arithmetic','hard','raw_or_pool','full_family','temperature_pool'])]
              .pivot(index=['dataset','model','seed','fold','train_cap'],columns='method',values='nll').reset_index())
comparison = recomputed.merge(previous,on=['dataset','model','seed','fold','train_cap'],validate='one_to_one')
comparisons = [('arithmetic','arithmetic_nll'),('hard','hard_nll'),('raw_or_pool','raw_or_pool_nll'),
               ('full_family','full_selected_nll'),('temperature_pool','temperature_pool_nll')]
max_differences = {name:float((comparison[name]-comparison[old]).abs().max()) for name,old in comparisons}
print('Maximum absolute differences against prior independent audit:',max_differences)
assert len(comparison)==160 and max(max_differences.values())<1e-8
write_json(REPORT_DIR/'08_PRIOR_AUDIT_AGREEMENT.json',
           {'n_matched_configs':len(comparison),'max_abs_nll_difference':max_differences,
            'note':'New results recomputed independently before this cross-check.'})


## 12. Generate and download the compact final audit ZIP

This ZIP contains verified **derived reports, figures, LaTeX and a checksum manifest**, but excludes the original 160 caches. Keep the original 20 MB combined evidence ZIP separately.

In [ ]:
final_zip, report_count = make_archive(REPORT_DIR)
print('AUDIT PACKAGE READY:',final_zip)
print('Report files included:',report_count)
print('Package size:',round(final_zip.stat().st_size/1048576,3),'MiB')
if DOWNLOAD_AUDIT_ZIP:
    try:
        from google.colab import files
        files.download(str(final_zip))
    except ImportError:
        print('Local runtime: download via file link or filesystem.')


## Results and interpretation guide

- **Strongest fixed intervention:** hard projection typically improves consistency yet can worsen joint test NLL. Report the 10-dataset paired estimates and Holm-adjusted exploratory uncertainty.
- **Competing baselines:** raw-or-pool, full-family selection, and temperature-calibrated pooling must all be reported; do not claim a complex repair wins when it does not.
- **Heterogeneity:** Diamonds has dataset-average hard-repair improvement. That observation does not establish a universal trigger, and several class-pair cells may have small counts.
- **Unfinished provenance:** historical model weight hashes, original OpenML revisions, exact transformed feature schemas for all 10 datasets, and clean pinned-environment retraining are *not* verified by the saved cache audit.
- **Manuscript next step:** use `07_MAIN_TABLE.tex`, the dataset-blocked results, the comparison figures, and `07_PAPER_RESULTS_AND_LIMITATIONS.md`. The notebook does not automatically rewrite or submit the research paper.